# BLIP Model Evaluation Pipeline

For the models to be considered for implementation, it will undergo comparitive evaluation against each other. Models that achieve an ROC-AUC of 0.75 or above will be considered, with Average Precision being the secondary factor. Additionally, a confusion matrix is created to visualise the performance of the models.

Chosen Models for Evaluation:
1. Salesforce/blip-itm-base-coco
2. Salesforce/blip-itm-large-coco
3. Salesforce/blip-itm-base-flickr
4. Salesforce/blip-itm-large-flickr
5. Salesforce/blip2-itm-vit-g-coco

In [2]:
!pip install torch pandas transformers scikit-learn matplotlib

In [3]:
# Importing Dependencies
import os
import gc
import json
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, BlipForImageTextRetrieval, Blip2ForImageTextRetrieval
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

C:\Users\isyra\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Variable Definitions

In [4]:
# Dataset CSV Path
csv_path = '../fakeddit_blip_50_50_test.csv'

# Image Directory
image_dir = '../fakeddit_images'

# Results Directory
results_dir = './blip_eval_results'

# Dataset Columns
image_col = 'id'
text_col = 'clean_title'
label_col = '6_way_label'


# Model List
models = ["./models/Salesforce-blip-itm-base-coco",
          "./models/Salesforce-blip-itm-large-coco",
          "./models/Salesforce-blip-itm-base-flickr",
          "./models/Salesforce-blip-itm-large-flickr",
          "./models/Salesforce-blip2-itm-vit-g-coco"]

# Utilizing CUDA if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device.upper()}")

Running on: CUDA


# Helper functions

## Labels to Binary

Changes the labels of 0 and 2 to 0 and 1, with 0 representing a true label and 1 representing the false connection label. This is crucial for the BLIP model, but also affects how the scores are determined.

In [5]:
def labels_to_binary(labels):
    return [1 if label == 2 else 0 for label in labels]

## Compute Metrics

Checks for any empty value predicts/results, computing and returning the ROC-AUC score, Average Precision Score, Valid and Invalid Samples and a Confusion Matrix.

In [6]:
def compute_metrics(y_true, mismatch_probs):
    # Filtering out none values in case
    valid_mask = [p is not None for p in mismatch_probs]
    valid_y_true = [y for y, m in zip(y_true, valid_mask) if m]
    valid_probs = [p for p, m in zip(mismatch_probs, valid_mask) if m]
    valid_count = sum(valid_mask)
    invalid_count = len(mismatch_probs) - valid_count

    # If no valid values (all none)
    if not valid_probs:
        return none

    y_true_arr = np.array(valid_y_true)
    probs_arr = np.array(valid_probs)

    # ROC-AUC
    roc_auc = roc_auc_score(y_true_arr, probs_arr)

    # Average Precision
    avg_prec = average_precision_score(y_true_arr, probs_arr)

    # Confusion Matrix at 0.5 threshold
    y_pred = (probs_arr >= 0.5).astype(int)
    cm = confusion_matrix(y_true_arr, y_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        "roc_auc": round(roc_auc, 4),
        "avg_precision": round(avg_prec, 4),
        "valid_samples": valid_count,
        "invalid_samples": invalid_count,
        "confusion_matrix": {"tn": int(tn), "fp": int(fp),
                             "fn": int(fn), "tp": int(tp)},
    }

## Print Confusion Matrix

Prints out the data into an actual confusion matrix for visualisation.

In [7]:
def print_confusion_matrix(cm_dict, model_name):
    # Printing the confusion matrix in console
    print(f"\nConfusion Matrix — {model_name.replace('Salesforce-', '')}")
    print(f"                 Predicted Real  Predicted Fake")
    print(f"  Actual Real       {cm_dict['tn']}              {cm_dict['fp']}")
    print(f"  Actual Fake       {cm_dict['fn']}              {cm_dict['tp']}")

# Loading in the dataset

In [8]:
# Loading in the Dataset
df = pd.read_csv(csv_path)
y_true = labels_to_binary(df[label_col])

# DataFrame for Storing Per Sample Results
df_results = df[[image_col, label_col]].copy()
df_results['binary_label'] = y_true

# Creating the directory for results
os.makedirs(results_dir, exist_ok = True)
summary_rows = []

# Multi-Model Inference Loop

This for loop tests the several models provided onto the adjusted fakeddit dataset. Notably, the results from the matching predictions provided by the models are inverted. This is due to the labelling of the dataset, as a 1 is indicative of a false connection whereas a higher score would have been indicative of a matching image text pair. By inverting the score, the predictions are made to be inline with the labelling of the dataset.

In [9]:
for model_path in models:
    # Retrieving model name for logs
    model_name = os.path.basename(model_path.rstrip('/'))
    is_blip2 = "blip2" in model_name.lower()

    print(f"Loading Model: {model_name}...")

    # Loading Model In
    processor = AutoProcessor.from_pretrained(model_path)
    if is_blip2:
        model = Blip2ForImageTextRetrieval.from_pretrained(model_path).to(device)
    else:
        model = BlipForImageTextRetrieval.from_pretrained(model_path).to(device)
    
    model.eval()
    mismatch_probs = []

    # Running inference (no gradient racking for memory and speed)
    with torch.no_grad():

        for index, row in tqdm(df.iterrows(), total = len(df), desc = model_name):

            text = row[text_col]
            post_id = str(row[image_col])
            image_path = os.path.join(image_dir, f"{post_id}.jpg")

            try:

                raw_image = Image.open(image_path).convert('RGB')

                # Processor handles the tokenization of text and pre-processing of image
                inputs = processor(images = raw_image,
                                   text = text,
                                   return_tensors = "pt",
                                   truncation = True).to(device)

                outputs = model(**inputs)

                if is_blip2:
                    # Blip2 Models return raw logits, in shape 1,1, therefore sigmoid use
                    match_prob = torch.nn.functional.sigmoid(outputs.logits_per_image).squeeze().item()
                else:
                    # Blip1 Models return logits for no_match and maatch, therefore softmax use
                    match_prob = torch.nn.functional.softmax(outputs.itm_score, dim = 1)[:, 1].item()

                # Invert Match probability to get mismatch probability (this is due to the label organisation)
                mismatch_probs.append(1.0 - match_prob)

            except Exception as e:
                print(e)
                mismatch_probs.append(None)
        
        # Saving per-sample model scores to results DF
        column_name = f"{model_name}_mismatch_score"
        df_results[column_name] = mismatch_probs
        df_results.to_csv(os.path.join(results_dir, "interim_scores.csv"), index = False)

        # Compute and display metrics
        metrics = compute_metrics(y_true, mismatch_probs)

        # Illustrating the metrics
        if metrics:
            cm = metrics['confusion_matrix']
            print(f"\nROC-AUC: {metrics['roc_auc']}")
            print(f"Average Precision: {metrics['avg_precision']}")
            print(f"Valid samples: {metrics['valid_samples']}")
            print(f"Invalid samples: {metrics['invalid_samples']}")
            print_confusion_matrix(cm, model_name)
    
            # Save per-model metrics as JSON
            json_path = os.path.join(results_dir, f"{model_name}_metrics.json")
            with open(json_path, 'w') as f:
                json.dump(metrics, f, indent = 2)

            # Saving a summarised result table
            summary_rows.append({
                "model": model_name,
                "roc_auc": metrics["roc_auc"],
                "avg_precision": metrics["avg_precision"],
                "valid_samples": metrics["valid_samples"],
                "invalid_samples": metrics["invalid_samples"],
                "tp": cm["tp"], "fp": cm["fp"],
                "fn": cm["fn"], "tn": cm["tn"],
            })
        else:
            print(f"{model_name} produced no valid results.")

        
        # Wipe the local GPU memory clean before the loop restarts for the next model
        print(f"\nClearing {model_name} from local GPU memory...")
        del model
        del processor
        if 'inputs' in locals(): del inputs
        if 'outputs' in locals(): del outputs
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"\nCache Cleared.")

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading Model: Salesforce-blip-itm-base-coco...


Salesforce-blip-itm-base-coco:  46%|██████████████████████▎                         | 929/2000 [00:32<00:40, 26.51it/s]C:\Users\isyra\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Salesforce-blip-itm-base-coco: 100%|███████████████████████████████████████████████| 2000/2000 [01:10<00:00, 28.23it/s]



  ROC-AUC: 0.7632
  Average Precision: 0.7653
  Valid samples: 2000
  Invalid samples: 0

Confusion Matrix — blip-itm-base-coco
                 Predicted Real  Predicted Fake
  Actual Real       617              383
  Actual Fake       229              771

Clearing Salesforce-blip-itm-base-coco from local GPU memory...

Cache Cleared.
Loading Model: Salesforce-blip-itm-large-coco...


Salesforce-blip-itm-large-coco:  46%|█████████████████████▊                         | 928/2000 [01:16<01:26, 12.40it/s]C:\Users\isyra\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Salesforce-blip-itm-large-coco: 100%|██████████████████████████████████████████████| 2000/2000 [02:44<00:00, 12.13it/s]


...


  ROC-AUC: 0.6148
  Average Precision: 0.6293
  Valid samples: 2000
  Invalid samples: 0

Confusion Matrix — blip-itm-base-flickr
                 Predicted Real  Predicted Fake
  Actual Real       87              913
  Actual Fake       21              979

Clearing Salesforce-blip-itm-base-flickr from local GPU memory...

Cache Cleared.
Loading Model: Salesforce-blip-itm-large-flickr...


Salesforce-blip-itm-large-flickr:  46%|████████████████████▉                        | 928/2000 [01:16<01:26, 12.42it/s]C:\Users\isyra\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Salesforce-blip-itm-large-flickr: 100%|████████████████████████████████████████████| 2000/2000 [02:43<00:00, 12.22it/s]



  ROC-AUC: 0.7654
  Average Precision: 0.7677
  Valid samples: 2000
  Invalid samples: 0

Confusion Matrix — blip-itm-large-flickr
                 Predicted Real  Predicted Fake
  Actual Real       765              235
  Actual Fake       349              651

Clearing Salesforce-blip-itm-large-flickr from local GPU memory...

Cache Cleared.
Loading Model: Salesforce-blip2-itm-vit-g-coco...


Salesforce-blip2-itm-vit-g-coco:  46%|█████████████████████▎                        | 929/2000 [03:58<04:32,  3.92it/s]C:\Users\isyra\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Salesforce-blip2-itm-vit-g-coco: 100%|█████████████████████████████████████████████| 2000/2000 [08:28<00:00,  3.93it/s]



  ROC-AUC: 0.7624
  Average Precision: 0.7258
  Valid samples: 2000
  Invalid samples: 0

Confusion Matrix — blip2-itm-vit-g-coco
                 Predicted Real  Predicted Fake
  Actual Real       1000              0
  Actual Fake       1000              0

Clearing Salesforce-blip2-itm-vit-g-coco from local GPU memory...

Cache Cleared.


# Printing of the results

In [12]:
# Reading and illustrating the summary results for all models together
if summary_rows:

    # Sorting and saving the summary results as dataframe and csv
    df_summary = pd.DataFrame(summary_rows).sort_values("roc_auc", ascending = False)
    summary_path = os.path.join(results_dir, "blip_model_comparison.csv")
    df_summary.to_csv(summary_path, index = False)

    # Printing the comparison
    print(f"\n{'='*60}")
    print("Model Comparison by ROC-AUC Score")
    print(f"{'='*60}")
    print(df_summary[["model", "roc_auc", "avg_precision"]].to_string(index = False))

# Save all per-sample scores for future reference
df_results.to_csv(os.path.join(results_dir, "blip_all_scores.csv"), index = False)
print(f"\nEvaluation complete. All results are stored in: {results_dir}/")


Model Comparison by ROC-AUC Score
                           model  roc_auc  avg_precision
Salesforce-blip-itm-large-flickr   0.7654         0.7677
   Salesforce-blip-itm-base-coco   0.7632         0.7653
 Salesforce-blip2-itm-vit-g-coco   0.7624         0.7258
  Salesforce-blip-itm-large-coco   0.7612         0.7662
 Salesforce-blip-itm-base-flickr   0.6148         0.6293

Evaluation complete. All results are stored in: ./blip_eval_results/
